Setup

In [2]:
from pinecone import Pinecone, ServerlessSpec 

pc = Pinecone(api_key="pcsk_RuCFD_JP37mTMRqQsA2KiVtQbvpQ4X2VNePDYDqQ5JZYnY3HqySFqkRzSGeztYGMn787q")
# Index 'datacamp-index' already exists, so we just connect to it

## Checking our indexes

In [2]:
pc.list_indexes()

[
    {
        "name": "datacamp-index",
        "metric": "cosine",
        "host": "datacamp-index-rjegbuo.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 1536,
        "deletion_protection": "disabled",
        "tags": null
    }
]

# Managing Indexes

## Connecting to the index

In [4]:
# Index 'datacamp-index' already exists, so we just connect to it
index = pc.Index('datacamp-index')


c:\Users\M S I\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
index.describe_index_stats()

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Sun, 30 Nov 2025 09:20:14 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '61',
                                    'x-pinecone-request-id': '2670499905405682672',
                                    'x-pinecone-request-latency-ms': '61'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}

Deleting Index

In [8]:
pc.delete_index('datacamp-index')
pc.list_indexes()


[]

# Vektor ingestion

Creating and connecting to an index

In [10]:
# Create the index first
pc.create_index(
	name='datacamp-index',
	dimension=1536,  # Adjust dimension based on your embedding model
	metric='cosine',
	spec=ServerlessSpec(
		cloud='aws',
		region='us-east-1'
	)
)

# Now connect to the index
index = pc.Index('datacamp-index')


Ingesting vectors

In [ ]:
vectors = [    
    {
        "id": "0",
        "values": [0.025525547564029694, ..., 0.0188823901116848]    
    # },        
    #      ...,    
    # {
        "id": "9",
        "values": [0.020712468773126602, ..., 0.006418442353606224]    
    },
]


Checking dimensionality

In [20]:
vector_dims = [len(vector['values']) == 1536 for vector in vectors if isinstance(vector, dict)]
all(vector_dims)

False

In [21]:
for i, vector in enumerate(vectors):
    if isinstance(vector, dict):
        print(i, len(vector.get("values", [])))


0 3
2 3


Upserting vectors

In [22]:
index.upsert(    
    vectors=vectors
)
index.describe_index_stats()

PineconeApiTypeError: Invalid type for variable '1'. Required value type is float and passed type was ellipsis at ['values'][1]

In [23]:
for i, v in enumerate(vectors):
    if ... in v["values"]:
        print("Error at index:", i)
        print(v)
        break


Error at index: 0
{'id': '0', 'values': [0.025525547564029694, Ellipsis, 0.0188823901116848]}


In [26]:
from openai import OpenAI
client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

def embed_text(text):
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return emb.data[0].embedding  # list of 1536 floats

vectors = []

for i, text in enumerate(my_texts):
    vector = embed_text(text)
    vectors.append({
        "id": str(i),
        "values": vector,
        "metadata": {"text": text}
    })


NameError: name 'my_texts' is not defined

In [28]:
vectors = [    
    {
        "id": "0",
        "values": [0.025525547564029694, ..., 0.0188823901116848],
        "metadata": {"genre": "productivity", "year": 2020}    
        },        
        ...,
]

Upserting vectors with metadata

In [30]:
# Clean vectors before upsert: ensure values are numeric and match expected_dim.
# Try to create embeddings from metadata["text"] if values are missing/invalid.
# Reuse embed_text if it was defined earlier; otherwise define a fallback using the existing client.
if "embed_text" not in globals():
    def embed_text(text):
        emb = client.embeddings.create(
            model="text-embedding-3-small",
            input=text
        )
        return emb.data[0].embedding  # list of floats

def is_valid_values(vals):
    if not isinstance(vals, list):
        return False
    if len(vals) != expected_dim:
        return False
    # allow floats and python numeric types (int -> will be accepted as numeric but cast below)
    return all(isinstance(x, (float, int)) for x in vals)

clean_vectors = []

for idx, vec in enumerate(vectors):
    if not isinstance(vec, dict):
        print(f"Skipping non-dict entry at index {idx}: {vec!r}")
        continue

    vals = vec.get("values")
    if is_valid_values(vals):
        # ensure floats
        vec["values"] = [float(x) for x in vals]
        clean_vectors.append(vec)
        continue

    # try to build embedding from metadata text if available
    metadata = vec.get("metadata", {})
    text = metadata.get("text") if isinstance(metadata, dict) else None
    if isinstance(text, str) and text.strip():
        emb = embed_text(text)
        if isinstance(emb, list) and len(emb) == expected_dim:
            new_vec = {"id": vec.get("id", str(idx)), "values": [float(x) for x in emb], "metadata": metadata}
            clean_vectors.append(new_vec)
            continue
        else:
            print(f"Generated embedding for id {vec.get('id')} has wrong dim: {len(emb) if isinstance(emb, list) else 'invalid'}")
            continue

    print(f"Skipping vector id {vec.get('id')} at index {idx}: invalid or incomplete values")

if not clean_vectors:
    raise ValueError("No valid vectors to upsert after cleaning. Check your input vectors or provide metadata['text'] to generate embeddings.")

# Upsert cleaned vectors
index.upsert(vectors=clean_vectors)
print("Upserted", len(clean_vectors), "vectors")
index.describe_index_stats()

Skipping vector id 0 at index 0: invalid or incomplete values
Skipping non-dict entry at index 1: Ellipsis


ValueError: No valid vectors to upsert after cleaning. Check your input vectors or provide metadata['text'] to generate embeddings.